In [ ]:
import zipfile
import os

zip_file_path = '/content/drive/MyDrive/Yelp-Photos.zip'
extract_dir = '/content/extracted_yelp_data'

# Create the extraction directory if it doesn't exist
os.makedirs(extract_dir, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Dataset extracted to: {extract_dir}")

Dataset extracted to: /content/extracted_yelp_data


In [ ]:
import json

json_file_path = '/content/extracted_yelp_data/Yelp Photos/extracted_tar/photos.json'

photos_data = []
with open(json_file_path, 'r') as f:
    for line in f:
        try:
            photos_data.append(json.loads(line))
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON on line: {line.strip()}. Error: {e}")
            # Optionally, you can log the line that caused the error or decide how to handle it
            continue


print(f"Loaded data for {len(photos_data)} photos.")
# You can inspect the first few entries to understand the data structure
# print(photos_data[:5])

Loaded data for 200100 photos.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import torchvision.transforms as transforms

class YelpPhotosDataset(Dataset):
    def __init__(self, photos_data, image_dir, transform=None):
        self.photos_data = photos_data
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.photos_data)

    def __getitem__(self, idx):
        img_info = self.photos_data[idx]
        img_filename = img_info['photo_id'] + '.jpg' # Assuming the images are in JPG format and named after their photo_id
        img_path = os.path.join(self.image_dir, img_filename)

        try:
            image = Image.open(img_path).convert('RGB') # Convert to RGB to handle potential grayscale images
            if self.transform:
                image = self.transform(image)
            return image
        except FileNotFoundError:
            print(f"Image file not found: {img_path}. Skipping.")
            # Return a dummy tensor or handle this case as appropriate for your training
            # For now, we will return None and filter them out later if needed
            return None
        except Exception as e:
            print(f"Error loading image {img_path}: {e}. Skipping.")
            return None

image_directory = '/content/extracted_yelp_data/Yelp Photos/extracted_tar/photos' # Assuming image files are in a 'photos' subdirectory

# We can start with resizing and converting to a tensor
image_transform = transforms.Compose([
    transforms.Resize((64, 64)), # Resize images to a fixed size
    transforms.ToTensor(), # Convert PIL Image to PyTorch Tensor
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Normalize pixel values to [-1, 1]
])

# Create the dataset
yelp_dataset = YelpPhotosDataset(photos_data, image_directory, transform=image_transform)

# Create the DataLoader
# batch_size = 32 # You can define your batch size
# train_dataloader = DataLoader(yelp_dataset, batch_size=batch_size, shuffle=True, num_workers=2) # Adjust num_workers based on your system

print("YelpPhotosDataset and DataLoader created.")


YelpPhotosDataset and DataLoader created.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings

class Block(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.silu = nn.SiLU()
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_ch)
        self.time_mlp = nn.Linear(time_emb_dim, out_ch)

    def forward(self, x, time_emb):
        h = self.silu(self.bn1(self.conv1(x)))
        h = h + self.time_mlp(time_emb)[:, :, None, None]
        h = self.silu(self.bn2(self.conv2(h)))
        return h

class UNet(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim):
        super().__init__()
        self.time_mlp = SinusoidalPositionEmbeddings(time_emb_dim)
        self.inc = Block(in_ch, 64, time_emb_dim)
        self.down1 = Block(64, 128, time_emb_dim)
        self.down2 = Block(128, 256, time_emb_dim)
        self.mid = Block(256, 256, time_emb_dim)
        self.up1 = Block(256 + 256, 128, time_emb_dim)
        self.up2 = Block(128 + 128, 64, time_emb_dim)
        self.outc = nn.Conv2d(64 + 64, out_ch, 1)

    def forward(self, x, t):
        time_emb = self.time_mlp(t)
        x1 = self.inc(x, time_emb)
        x2 = self.down1(F.max_pool2d(x1, 2), time_emb)
        x3 = self.down2(F.max_pool2d(x2, 2), time_emb)
        x = self.mid(F.max_pool2d(x3, 2), time_emb)
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        x = torch.cat([x, x3], dim=1)
        x = self.up1(x, time_emb)
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        x = torch.cat([x, x2], dim=1)
        x = self.up2(x, time_emb)
        x = F.interpolate(x, scale_factor=2, mode='nearest')
        x = torch.cat([x, x1], dim=1)
        output = self.outc(x)
        return output

# Define the DDIM class which will incorporate the UNet
class DDIM(nn.Module):
    def __init__(self, unet, timesteps=1000, beta_start=1e-4, beta_end=0.02):
        super().__init__()
        self.unet = unet
        self.timesteps = timesteps

        # Define the variance schedule (linear schedule)
        betas = torch.linspace(beta_start, beta_end, timesteps)
        alphas = 1. - betas
        alphas_cumprod = torch.cumprod(alphas, axis=0)
        alphas_cumprod_prev = F.pad(alphas_cumprod[:-1], (1, 0), value=1.0)
        sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
        sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
        sqrt_one_minus_alphas_cumprod = torch.sqrt(1. - alphas_cumprod)
        posterior_variance = betas * (1. - alphas_cumprod_prev) / (1. - alphas_cumprod)

        self.register_buffer('betas', betas)
        self.register_buffer('alphas_cumprod', alphas_cumprod)
        self.register_buffer('sqrt_recip_alphas', sqrt_recip_alphas)
        self.register_buffer('sqrt_alphas_cumprod', sqrt_alphas_cumprod)
        self.register_buffer('sqrt_one_minus_alphas_cumprod', sqrt_one_minus_alphas_cumprod)
        self.register_buffer('posterior_variance', posterior_variance)

    def forward_diffusion(self, x_0, t, noise):
        # Forward diffusion process (adding noise to the image)
        q_sample = self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1) * x_0 + \
                   self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1) * noise
        return q_sample

    def backward_diffusion(self, x_t, t):
        # Backward diffusion process (denoising the image using the UNet)
        # This is the core of the DDIM sampling step
        predicted_noise = self.unet(x_t, t)

        # Calculate the predicted x_0
        pred_x_0 = (x_t - self.sqrt_one_minus_alphas_cumprod[t].view(-1, 1, 1, 1) * predicted_noise) / \
                   self.sqrt_alphas_cumprod[t].view(-1, 1, 1, 1)

        # Calculate the direction pointing to x_t
        direction_pointing_to_x_t = torch.sqrt(1. - self.alphas_cumprod[t].view(-1, 1, 1, 1)) * predicted_noise

        # DDIM sampling equation
        x_prev = self.sqrt_alphas_cumprod[t-1].view(-1, 1, 1, 1) * pred_x_0 + direction_pointing_to_x_t # Assuming sigma_t = 0 for deterministic sampling

        return x_prev, pred_x_0

    @torch.no_grad()
    def sample(self, shape, device):
        # DDIM sampling process
        batch_size = shape[0]
        img = torch.randn(shape, device=device) # Start with random noise

        for i in reversed(range(1, self.timesteps)):
            t = torch.full((batch_size,), i, device=device, dtype=torch.long)
            img, _ = self.backward_diffusion(img, t)

        return img

In [ ]:
# Instantiate the UNet and DDIM models
# Adjust input channels (3 for RGB images) and output channels (3 for RGB images)
# Adjust time_emb_dim based on your preference
unet = UNet(in_ch=3, out_ch=3, time_emb_dim=256)
ddim_model = DDIM(unet, timesteps=1000)

print("DDIM model implemented.")


DDIM model implemented.


In [ ]:
import torch.optim as optim
import os

# Define training parameters
batch_size = 32 # You can define your batch size
epochs =10 # Number of training epochs
learning_rate = 0.0001 # Learning rate for the optimizer
timesteps = ddim_model.timesteps # Number of diffusion timesteps
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Custom collate function to filter out None values
def custom_collate_fn(batch):
    batch = [item for item in batch if item is not None]
    if len(batch) == 0:
        return None # Return None for empty batches, to be handled in the training loop
    return torch.stack(batch, dim=0)


# Create the DataLoader
train_dataloader = DataLoader(yelp_dataset, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=custom_collate_fn) # Adjust num_workers based on your system

# Move model to device
ddim_model.to(device)

# Define optimizer
optimizer = optim.Adam(ddim_model.parameters(), lr=learning_rate)

# Define loss function (Mean Squared Error between predicted noise and actual noise)
criterion = nn.MSELoss()

# Create directories for saving logs and checkpoints
logs_dir = './logs'
checkpoint_dir = './checkpoints'
os.makedirs(logs_dir, exist_ok=True)
os.makedirs(checkpoint_dir, exist_ok=True)

print("Training parameters and setup complete.")

Training parameters and setup complete.


In [ ]:
import torch
import os
import time
import numpy as np

# Training loop
print("Starting training...")
for epoch in range(epochs):
    ddim_model.train()
    running_loss = 0.0
    start_time = time.time()

    for i, images in enumerate(train_dataloader):
        # Filter out None values in case of image loading errors
        images = [img for img in images if img is not None]
        if len(images) == 0:
            continue
        images = torch.stack(images, dim=0).to(device)

        optimizer.zero_grad()

        # Sample a random timestep for each image in the batch
        t = torch.randint(0, timesteps, (images.shape[0],), device=device).long()

        # Generate random noise
        noise = torch.randn_like(images)

        # Apply forward diffusion
        noisy_images = ddim_model.forward_diffusion(images, t, noise)

        # Predict the noise using the UNet
        predicted_noise = ddim_model.unet(noisy_images, t)

        # Calculate the loss
        loss = criterion(predicted_noise, noise)

        # Backpropagate and update weights
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # Print loss every a few batches
        if (i + 1) % 100 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(train_dataloader)}], Loss: {loss.item():.4f}")

    epoch_loss = running_loss / len(train_dataloader)
    end_time = time.time()
    epoch_duration = end_time - start_time
    print(f"Epoch [{epoch+1}/{epochs}] finished, Average Loss: {epoch_loss:.4f}, Duration: {epoch_duration:.2f} seconds")

    # Save checkpoint
    if (epoch + 1) % 10 == 0 or (epoch + 1) == epochs:
        checkpoint_path = os.path.join(checkpoint_dir, f'ddim_checkpoint_epoch_{epoch+1}.pth')
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': ddim_model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': epoch_loss,
        }, checkpoint_path)
        print(f"Checkpoint saved to {checkpoint_path}")

    # Save logs
    log_path = os.path.join(logs_dir, 'training_log.txt')
    with open(log_path, 'a') as f:
        f.write(f"Epoch [{epoch+1}/{epochs}], Average Loss: {epoch_loss:.4f}, Duration: {epoch_duration:.2f} seconds\n")

print("Training complete.")

Starting training...
Error loading image /content/extracted_yelp_data/Yelp Photos/extracted_tar/photos/feUGw0P5byOq4U40C77tyQ.jpg: cannot identify image file '/content/extracted_yelp_data/Yelp Photos/extracted_tar/photos/feUGw0P5byOq4U40C77tyQ.jpg'. Skipping.
Epoch [1/10], Step [100/6254], Loss: 0.0658
Error loading image /content/extracted_yelp_data/Yelp Photos/extracted_tar/photos/rrfwGSwt3eHxxypfu5PGTA.jpg: cannot identify image file '/content/extracted_yelp_data/Yelp Photos/extracted_tar/photos/rrfwGSwt3eHxxypfu5PGTA.jpg'. Skipping.
Error loading image /content/extracted_yelp_data/Yelp Photos/extracted_tar/photos/iX-8Xm2G7meRHUg8qhoL1A.jpg: cannot identify image file '/content/extracted_yelp_data/Yelp Photos/extracted_tar/photos/iX-8Xm2G7meRHUg8qhoL1A.jpg'. Skipping.
Error loading image /content/extracted_yelp_data/Yelp Photos/extracted_tar/photos/0TpeNZPs3Gu8s30KVXudcg.jpg: cannot identify image file '/content/extracted_yelp_data/Yelp Photos/extracted_tar/photos/0TpeNZPs3Gu8s30KVX